## RNNs
The encoder tries to capture the entire meaning of the input sequence in the final hidden state (embedding vector).

The decoder takes this state to start generating the translated sentence, one word at a time. 
It also updates its hidden state at each step, which is supposed to carry the context necessary for the next word prediction.
 
The limitation is that RNN cannot directly access earlier hidden states from the encoder during the decoding phase - loss of context.

Bahdanau attention - decoder selectively accesses different parts of the input sentence at each decoding step. The importance of each token is determined by the attention weight. 



## Self Attention
It allow each position in the input to consider (attend to) all other positions in the same sequence, when computing the representation of a sequence. 

The goal is to obtain the context vector v<sup>(i)</sup> for each element x<sup>(i)</sup> in the input sentence - enriched embedding vector. 

This allows to understand relationships and relevance of words ina a sentence to each other. 

In [1]:
import torch

torch.manual_seed(101)

text_length = 6
embed_dim = 3
inputs = torch.rand(text_length, embed_dim)
print('Inputs:')
print(inputs)
print()

# compute intermediate values - attention scores (w)
query = inputs[1] # example for the second token
attn_scores_2 = torch.empty(text_length) # dla każdego tokenu
for i, xi in enumerate(inputs):
    attn_scores_2[i] = torch.dot(xi, query) # similarity

# Normalization
attn_weights_2 = attn_scores_2 / attn_scores_2.sum()
print(attn_weights_2)
print(attn_weights_2.sum())
print()

# Softmax normalization
# Better at managing extreme cases, better gradient properties, weights always positive
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print('Attention weights:')
print(attn_weights_2)
print(attn_weights_2.sum())
print()

# Calculating the context vector for the second token
context_vec_2 = torch.zeros(embed_dim)
for i, xi in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * xi
print('Context vector z1 for x1:')
print(context_vec_2)

Inputs:
tensor([[0.1980, 0.4503, 0.0909],
        [0.8872, 0.2894, 0.0186],
        [0.9095, 0.3406, 0.4309],
        [0.7324, 0.4776, 0.0716],
        [0.5834, 0.7521, 0.7649],
        [0.1443, 0.7152, 0.3953]])

tensor([0.0774, 0.2193, 0.2299, 0.1986, 0.1886, 0.0862])
tensor(1.)

Attention weights:
tensor([0.1137, 0.1997, 0.2083, 0.1840, 0.1768, 0.1177])
tensor(1.)

Context vector z1 for x1:
tensor([0.6439, 0.4849, 0.2987])


In [2]:
# Computing weights for all input tokens
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
context_vectors = attn_weights @ inputs
print(context_vectors[1])

tensor([0.6439, 0.4849, 0.2987])


## Attention with trainable weights
A.k.a. scaled dot-product attention

We introduce three trainable matrices:

W<sub>q</sub> - projects the embedded input token x<sup>(i)</sup> into query

W<sub>k</sub> - into key

W<sub>v</sub> - into vale

In [3]:
d_in = inputs.shape[1]
d_out = 2 # usually the same as d_in
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

queries = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

attn_scores = queries @ keys.T
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1]**0.5, # scaling factor to keep variance close to 1
    dim=-1
)
context_vec = attn_weights @ values
print(context_vec)

tensor([[0.6448, 0.7539],
        [0.6554, 0.7673],
        [0.6638, 0.7768],
        [0.6579, 0.7705],
        [0.6727, 0.7875],
        [0.6558, 0.7674]])


Matrix multiplication can be optimizet by utilizing nn.Linear, which performs it effectively, when bias units are disabled. 

## Casual Attention
A.k.a masked attention. It restricts a model to only consider previous and current inputsin a sequence. 

We mask out the future tokens, which come after the current one in the input text.  

In [4]:
context_length = inputs.shape[0]
mask = torch.triu( 
    torch.ones(context_length, context_length, dtype=torch.bool),
    diagonal=1
)
print(mask)
print()

# Softmax transforma 0 to positive values, so we use -inf
masked_scores = attn_scores.masked_fill(mask, -torch.inf)
masked_weights = torch.softmax(
    masked_scores / keys.shape[-1]**.5, 
    dim=1
)
print(masked_weights)

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4262, 0.5738, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2379, 0.3425, 0.4196, 0.0000, 0.0000, 0.0000],
        [0.1874, 0.2589, 0.3058, 0.2479, 0.0000, 0.0000],
        [0.1236, 0.1938, 0.2495, 0.1828, 0.2502, 0.0000],
        [0.1270, 0.1703, 0.1994, 0.1639, 0.1986, 0.1409]])


## Dropout
Typically, it's applied after calculating the attention weights or affter applying them to the value vector.

In [5]:
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example)) # the remaining elements are scaled up

tensor([[0., 0., 0., 0., 2., 2.],
        [0., 2., 2., 2., 0., 0.],
        [2., 0., 2., 0., 2., 0.],
        [0., 2., 2., 2., 0., 0.],
        [0., 0., 0., 2., 2., 2.],
        [2., 0., 2., 2., 0., 0.]])


## Multi-Head Attention
It stacks multiple instances of CasualAttention and concatenates the resultant context vectors along the column dimension. 

In [9]:
from self_attention import MultiheadAttentionWrapper
from multihead import MultiheadAttention

batch = torch.stack([inputs, inputs], dim=0)

conext_length = batch.shape[1] # number of tokens
print(context_length)
d_in = 3
d_out = 2
n_heads = 2

torch.manual_seed(101)
mhaw = MultiheadAttentionWrapper(d_in, d_out, context_length, 0.0, n_heads)
mha = MultiheadAttention(d_in, n_heads * d_out, context_length, 0.0, n_heads) # d_out oznacza całkowity wymiar wyjścia, niezależnie od liczby głów.

context_vectors_1 = mhaw(batch)
context_vectors_2 = mha(batch)
print(context_vectors_1.shape) # batch, context, heads x d_out
print(context_vectors_2.shape) # inna interpretacja d_out + out_proj

6
torch.Size([2, 6, 4])
torch.Size([2, 6, 4])
